In [ ]:
"数据集的下载"

import hashlib
import os
import tarfile
import zipfile
import requests

# 数据集统一托管的云根路径
DATA_URL = 'http://d2l-data.s3-accelerate.amazonaws.com/'
DATA_HUB = dict()
rootpath = "C:/Users/17934/Desktop/MachineLearning/Program/Kaggle_Exercise/DATA"

def download(name, cache_dir=os.path.join('..', 'data')):
    """
    从DATA_HUB下载数据集，本地缓存复用
    param name: DATA_HUB中注册的数据集名称
    param cache_dir: 本地缓存文件夹，默认 ../data
    return: 本地缓存压缩包完整路径
    """

    # 校验数据集是否已注册，不存在直接报错
    assert name in DATA_HUB, f"数据集 [{name}] 未在DATA_HUB中注册！"
    # 取出该数据集的云端路径、标准哈希值
    relative_url, standard_sha1 = DATA_HUB[name]
    full_url = DATA_URL + relative_url

    # 创建缓存目录，exist_ok=True：目录已存在不会报错
    os.makedirs(cache_dir, exist_ok=True)
    # 截取url末尾文件名，拼接成本地完整缓存路径
    file_name = full_url.split('/')[-1]
    local_file_path = os.path.join(cache_dir, file_name)

    # ========== 缓存命中逻辑：本地存在文件则校验哈希 ==========
    if os.path.exists(local_file_path):
        # 创建sha1哈希计算器
        sha1_calc = hashlib.sha1()
        # 二进制只读打开文件，分块读取（每次1MB，防止大文件占满内存）
        with open(local_file_path, 'rb') as f:
            while True:
                chunk = f.read(1048576)  # 1048576 byte = 1MB
                if not chunk:
                    break
                sha1_calc.update(chunk)
        # 对比计算出的哈希 和 官方标准哈希
        file_sha1 = sha1_calc.hexdigest()
        if file_sha1 == standard_sha1:
            print(f"缓存命中：复用本地文件 {local_file_path}")
            return local_file_path
        else:
            print(f"本地文件哈希不匹配，文件损坏/版本错误，重新下载...")

    # ========== 缓存失效/无本地文件：发起网络下载 ==========
    print(f"开始下载数据集：{full_url} -> {local_file_path}")
    # stream=True 流式下载，不一次性把整个文件载入内存
    response = requests.get(full_url, stream=True, verify=True)
    # 二进制写入本地
    with open(local_file_path, 'wb') as f:
        f.write(response.content)
    return local_file_path

# 4. 下载+自动解压工具函数
def download_extract(name, folder=None):
    """
    下载数据集并自动解压zip/tar/tar.gz
    :param name: 数据集注册名
    :param folder: 可选，指定解压后需要返回的子文件夹名
    :return: 解压后的数据集文件夹路径
    """
    # 第一步：调用download拿到本地压缩包路径
    zip_tar_path = download(name)
    # 获取压缩包所在目录
    base_cache_dir = os.path.dirname(zip_tar_path)
    # 分离文件名与后缀，判断压缩包类型
    data_root, ext = os.path.splitext(zip_tar_path)

    # 根据后缀选择解压工具
    if ext == '.zip':
        archive = zipfile.ZipFile(zip_tar_path, 'r')
    elif ext in ('.tar', '.gz'):
        archive = tarfile.open(zip_tar_path, 'r')
    else:
        # 非zip/tar直接抛出断言错误
        assert False, "仅支持 .zip / .tar / .tar.gz 格式压缩包"
    
    # 全部解压到缓存目录
    archive.extractall(base_cache_dir)
    archive.close()

    # 如果指定folder，返回子文件夹路径；否则返回解压根目录
    if folder:
        return os.path.join(base_cache_dir, folder)
    else:
        return data_root

# 5. 批量下载所有注册数据集
def download_all():
    """遍历DATA_HUB，一键下载全部数据集"""
    for dataset_name in DATA_HUB:
        download(dataset_name)